In [5]:
"""
HorusEye Q1 niveau 2 — Trouver des candidats "personne au sol" dans RefCOCO
À lancer sur TA copie de RefCOCO (là où sont refs(unc).p et instances.json).

Objectif : présélectionner des images de personnes en posture NON-debout
(au sol, assise, penchée) pour que TU les annotes manuellement à la main,
comme tu l'as fait pour tes 50 premières images.

Le script NE crée PAS d'annotations — il liste des CANDIDATS à vérifier.
Tu regardes chaque image et tu confirmes/corriges la posture toi-même.
"""

import pickle
import json
import os
from collections import defaultdict

# ============================================================
# CONFIG — ajuste ces chemins vers ta copie de RefCOCO
# ============================================================
REFCOCO_DIR = "/Volumes/TheDay/thedayproject/Cours_Udem/cours_VL/HorusEye/Horus_dev/rq1_datasets/refcoco/refer/data/refcoco"              # dossier contenant refs(unc).p
INSTANCES = "/Volumes/TheDay/thedayproject/Cours_Udem/cours_VL/HorusEye/Horus_dev/rq1_datasets/refcoco/refer/data/refcoco/instances.json" # annotations COCO
COCO_IMAGES = "/Volumes/TheDay/thedayproject/Cours_Udem/cours_VL/HorusEye/Horus_dev/rq1_datasets/coco"       # images (pour vérifier existence)
OUTPUT = "./lying_candidates.json"
ALREADY_ANNOTATED = "/Volumes/TheDay/thedayproject/Cours_Udem/cours_VL/HorusEye/Horus_dev/RQ3_health_vqa/rq3_health_annotations.json"  # pour EXCLURE tes 50 déjà faites

# Mots-clés d'expressions indiquant une posture NON-debout
POSTURE_KEYWORDS = {
    "lying":    ["lying", "laying", "lie down", "lies", "reclining", "sprawled"],
    "on_ground":["on the ground", "on the floor", "on ground", "on floor",
                 "on the grass", "on the beach", "on the bed", "on a bed",
                 "on the couch", "on the sofa"],
    "sleeping": ["sleeping", "asleep", "napping", "passed out"],
    "sitting":  ["sitting", "seated", "sits", "crouching", "kneeling", "squatting"],
    "bent":     ["bending", "bent over", "leaning", "crouched", "hunched"],
}

MAX_PER_CATEGORY = 40  # combien de candidats max par catégorie à sortir

# ============================================================
# 1. CHARGER REFCOCO
# ============================================================
def load_refcoco():
    ref_file = os.path.join(REFCOCO_DIR, "refs(unc).p")
    if not os.path.exists(ref_file):
        print(f"⚠️  {ref_file} introuvable. Contenu de {REFCOCO_DIR}:")
        if os.path.isdir(REFCOCO_DIR):
            for f in os.listdir(REFCOCO_DIR): print("   ", f)
        raise SystemExit("Ajuste REFCOCO_DIR.")
    with open(ref_file, "rb") as f:
        refs = pickle.load(f)
    print(f"✓ {len(refs)} référents chargés")

    with open(INSTANCES) as f:
        inst = json.load(f)
    # bbox par annotation id
    ann_bbox = {a["id"]: a["bbox"] for a in inst["annotations"]}
    # dims image par image id
    img_info = {i["id"]: (i["width"], i["height"], i["file_name"])
                for i in inst["images"]}
    # ne garder que la catégorie "person"
    person_cat = [c["id"] for c in inst["categories"] if c["name"] == "person"]
    person_cat = person_cat[0] if person_cat else 1
    person_anns = {a["id"] for a in inst["annotations"] if a["category_id"] == person_cat}
    return refs, ann_bbox, img_info, person_anns

In [6]:
# ============================================================
# 2. FILTRER LES CANDIDATS
# ============================================================
def find_candidates():
    refs, ann_bbox, img_info, person_anns = load_refcoco()

    # exclure les images déjà annotées (tes 50)
    already = set()
    if os.path.exists(ALREADY_ANNOTATED):
        d = json.load(open(ALREADY_ANNOTATED))
        for s in d.get("samples", []):
            already.add(str(s.get("filename", "")).replace(".jpg", ""))
        print(f"✓ {len(already)} images déjà annotées seront exclues")

    candidates = defaultdict(list)
    seen = set()

    for ref in refs:
        ann_id = ref["ann_id"]
        # seulement les personnes
        if ann_id not in person_anns:
            continue
        # toutes les expressions de ce référent
        sentences = [s["sent"].lower() for s in ref["sentences"]]
        text = " ".join(sentences)

        # matcher une catégorie de posture
        matched_cat = None
        for cat, kws in POSTURE_KEYWORDS.items():
            if any(kw in text for kw in kws):
                matched_cat = cat
                break
        if not matched_cat:
            continue

        image_id = ref["image_id"]
        if image_id not in img_info:
            continue
        w, h, fname = img_info[image_id]
        base = fname.replace(".jpg", "").split("_")[-1]  # ex COCO_train2014_000000123 -> 000000123
        if base in already or base in seen:
            continue

        bbox = ann_bbox.get(ann_id)
        if not bbox:
            continue
        bw, bh = bbox[2], bbox[3]
        ratio = bw / bh if bh > 0 else 0

        candidates[matched_cat].append({
            "image_id": image_id,
            "file_name": fname,
            "ann_id": ann_id,
            "bbox": bbox,
            "bbox_ratio_wh": round(ratio, 2),
            "expressions": sentences[:3],
            "posture_hint": matched_cat,
            "posture_TO_ANNOTATE": ""   # ← TOI tu remplis après avoir regardé l'image
        })
        seen.add(base)

    return candidates

# ============================================================
# 3. SORTIE
# ============================================================
def main():
    cands = find_candidates()

    print("\n" + "=" * 55)
    print("CANDIDATS TROUVÉS (à vérifier visuellement)")
    print("=" * 55)
    total = 0
    output = []
    for cat, items in cands.items():
        # prioriser ceux dont la bbox est "paysage" (ratio>1) = plus probablement au sol
        items.sort(key=lambda x: -x["bbox_ratio_wh"])
        items = items[:MAX_PER_CATEGORY]
        print(f"\n  [{cat}] : {len(items)} candidats")
        for it in items[:3]:
            print(f"    {it['file_name']} | ratio={it['bbox_ratio_wh']} | \"{it['expressions'][0][:50]}\"")
        output.extend(items)
        total += len(items)

    with open(OUTPUT, "w") as f:
        json.dump({
            "note": "CANDIDATS à annoter manuellement. Regarde chaque image, "
                    "remplis 'posture_TO_ANNOTATE' avec STANDING/SITTING/LYING. "
                    "Le 'posture_hint' est une SUGGESTION basée sur le texte, PAS la vérité.",
            "candidates": output
        }, f, indent=2)

    print(f"\n✓ {total} candidats écrits dans {OUTPUT}")
    print("\nÉTAPE SUIVANTE (toi) :")
    print("  1. Ouvre chaque image candidate")
    print("  2. Regarde la VRAIE posture de la personne (bbox fournie)")
    print("  3. Remplis 'posture_TO_ANNOTATE' : STANDING / SITTING / LYING")
    print("  4. Garde ~15-20 vrais LYING pour équilibrer ton dataset")
    print("\n⚠️  Le 'posture_hint' vient du TEXTE, il peut être faux.")
    print("    Ex: 'sitting on the ground' peut être assis OU allongé.")
    print("    C'est TON œil sur l'image qui décide, pas le mot-clé.")

if __name__ == "__main__":
    main()

✓ 50000 référents chargés
✓ 50 images déjà annotées seront exclues

CANDIDATS TROUVÉS (à vérifier visuellement)

  [on_ground] : 40 candidats
    COCO_train2014_000000095676.jpg | ratio=2.84 | "player sliding"
    COCO_train2014_000000018930.jpg | ratio=2.79 | "guy underneath that is falling"
    COCO_train2014_000000552116.jpg | ratio=2.78 | "man on ground in black and yellow"

  [sitting] : 40 candidats
    COCO_train2014_000000491249.jpg | ratio=3.31 | "person sitting top middle"
    COCO_train2014_000000377570.jpg | ratio=1.95 | "woman sitting in a chair"
    COCO_train2014_000000190087.jpg | ratio=1.89 | "weird but theyre both boxed"

  [bent] : 40 candidats
    COCO_train2014_000000515550.jpg | ratio=2.76 | "man at bottom with black shirt and glasses"
    COCO_train2014_000000078553.jpg | ratio=2.27 | "person on left leaning over only partially visible"
    COCO_train2014_000000122001.jpg | ratio=1.98 | "woman leaning back"

  [lying] : 40 candidats
    COCO_train2014_00000045397

HorusEye Q1 level 2 - Unified posture dataset merge
Combines:
  - rq3_health_annotations.json  (50 originals: gt_bbox, posture)
  - lying_candidates.json annotated (170 new: bbox, posture_TO_ANNOTATE)
Outputs a single JSON + final class distribution.

Run where both files are located.

In [7]:
import json
import os
from collections import Counter

RQ3_FILE = "/Volumes/TheDay/thedayproject/Cours_Udem/cours_VL/HorusEye/Horus_dev/RQ3_health_vqa/rq3_health_annotations.json"
CANDIDATES_FILE = "/Volumes/TheDay/thedayproject/Cours_Udem/cours_VL/HorusEye/Horus_dev/Q1_scene_understanding/lying_candidates.json"
OUTPUT = "/Volumes/TheDay/thedayproject/Cours_Udem/cours_VL/HorusEye/Horus_dev/outputs/posture_dataset_unified.json"

VALID_POSTURES = {"STANDING", "SITTING", "LYING"}

def normalize_filename(name):
    """Normalize every name to COCO_train2014_XXXXXXXXXXXX.jpg for consistency."""
    base = name.replace(".jpg", "")
    digits = base.split("_")[-1]
    num = int(digits)
    return f"COCO_train2014_{num:012d}.jpg"

unified = []
seen = set()

# The 50 originals
d1 = json.load(open(RQ3_FILE))
n1 = 0
for s in d1["samples"]:
    posture = s.get("posture", "").strip().upper()
    if posture not in VALID_POSTURES:
        continue
    fname = normalize_filename(s["filename"])
    if fname in seen:
        continue
    unified.append({
        "file_name": fname,
        "bbox": s["gt_bbox"],            # [x, y, w, h]
        "posture": posture,
        "expression": s.get("expression", ""),
        "source": "rq3_original_50",
    })
    seen.add(fname)
    n1 += 1

# The 170 newly annotated
d2 = json.load(open(CANDIDATES_FILE))
cands = d2.get("candidates", d2) if isinstance(d2, dict) else d2
n2 = 0
skipped_empty = 0
for c in cands:
    posture = c.get("posture_TO_ANNOTATE", "").strip().upper()
    if posture not in VALID_POSTURES:
        skipped_empty += 1
        continue
    fname = normalize_filename(c["file_name"])
    if fname in seen:
        continue
    exprs = c.get("expressions", [])
    unified.append({
        "file_name": fname,
        "bbox": c["bbox"],               # [x, y, w, h]
        "posture": posture,
        "expression": exprs[0] if exprs else "",
        "source": "lying_candidates_170",
    })
    seen.add(fname)
    n2 += 1

# Distribution
dist = Counter(x["posture"] for x in unified)

print("=" * 50)
print("UNIFIED POSTURE DATASET")
print("=" * 50)
print(f"  From rq3_original_50    : {n1} images")
print(f"  From lying_candidates   : {n2} images")
print(f"  Unannotated candidates skipped : {skipped_empty}")
print(f"  TOTAL unique            : {len(unified)}")
print()
print("  Final posture distribution:")
for p in ["STANDING", "SITTING", "LYING"]:
    n = dist.get(p, 0)
    pct = 100 * n / len(unified) if unified else 0
    bar = "#" * int(pct / 2)
    print(f"    {p:10} : {n:3}  ({pct:4.1f}%) {bar}")
print()

if dist:
    mx, mn = max(dist.values()), min(dist.values())
    print(f"  Imbalance ratio (max/min) : {mx/mn:.1f}x")
    if mx / mn > 3:
        print("  -> use macro F1 for evaluation")

with open(OUTPUT, "w") as f:
    json.dump({
        "metadata": {
            "description": "HorusEye Q1 level 2 - unified posture dataset",
            "total": len(unified),
            "distribution": dict(dist),
            "sources": {"rq3_original": n1, "lying_candidates": n2},
        },
        "samples": unified,
    }, f, indent=2)

print(f"\nDone. Written: {OUTPUT}")
print("\nNEXT STEP: Qwen3-VL calibration run")
print("  - full + cropped (bbox provided)")
print("  - clean/fog/smoke")
print("  - accuracy + ECE/AUROC, global AND per class (LYING priority)")
print("\nNote: make sure degraded images (clean/fog/smoke) exist")
print("  for ALL these images, not only the original 50.")
print("  The 170 new ones must go through your degradation pipeline.")

UNIFIED POSTURE DATASET
  From rq3_original_50    : 50 images
  From lying_candidates   : 170 images
  Unannotated candidates skipped : 0
  TOTAL unique            : 220

  Final posture distribution:
    STANDING   :  68  (30.9%) ###############
    SITTING    :  96  (43.6%) #####################
    LYING      :  56  (25.5%) ############

  Imbalance ratio (max/min) : 1.7x

Done. Written: /Volumes/TheDay/thedayproject/Cours_Udem/cours_VL/HorusEye/Horus_dev/outputs/posture_dataset_unified.json

NEXT STEP: Qwen3-VL calibration run
  - full + cropped (bbox provided)
  - clean/fog/smoke
  - accuracy + ECE/AUROC, global AND per class (LYING priority)

Note: make sure degraded images (clean/fog/smoke) exist
  for ALL these images, not only the original 50.
  The 170 new ones must go through your degradation pipeline.


## HorusEye Q1 level 2 - Degrade the 170 new posture images
# Applies fog + smoke (veils) on the newly selected COCO images,
# with the same pipeline (Koschmieder + Perlin), severity=0.5.

# Generates: clean/ fog/ smoke/  (same file names)
# Full consistency with RQ1 / Q1 level 1 and RefCOCO-Degraded.

# Run where the original COCO images are located.

In [8]:
import os
import json
import numpy as np
import cv2

# ============================================================
# CONFIG - adjust these paths
# ============================================================
UNIFIED_DATASET = "/Volumes/TheDay/thedayproject/Cours_Udem/cours_VL/HorusEye/Horus_dev/outputs/posture_dataset_unified.json"  # the 220-image file
COCO_SOURCE = "/Volumes/TheDay/thedayproject/Cours_Udem/cours_VL/HorusEye/Horus_dev/rq1_datasets/coco/images/train2014"                     # original COCO images
OUTPUT_ROOT = "/Volumes/TheDay/thedayproject/Cours_Udem/cours_VL/HorusEye/Horus_dev/outputs/posture_degraded"                   # output: clean/ fog/ smoke/
SEVERITY = 0.5
CONDITIONS = ["clean", "fog", "smoke"]    

# ============================================================
# Degradation pipeline (same as RQ1 - fog Koschmieder + smoke Perlin)
# ============================================================
class DegradationPipeline:
    def __init__(self, severity=0.5):
        self.severity = np.clip(severity, 0., 1.)

    def add_fog(self, image, depth_map=None):
        f = image.astype(np.float32) / 255.
        h, w = image.shape[:2]
        if depth_map is None:
            depth_map = np.tile(np.linspace(1., 0., h).reshape(h, 1), (1, w))
            n = cv2.GaussianBlur(np.random.rand(h, w).astype(np.float32) * .15, (51, 51), 0)
            depth_map = np.clip(depth_map + n, 0, 1)
        A = .95; beta = self.severity * 2.
        t = np.exp(-beta * depth_map); t = np.stack([t] * 3, -1)
        return np.clip((f * t + A * (1 - t)) * 255, 0, 255).astype(np.uint8)

    def add_smoke(self, image):
        h, w = image.shape[:2]
        f = image.astype(np.float32) / 255.
        sm = self._smoke_tex(h, w); col = np.array([.9, .9, .92])
        a = np.stack([sm * self.severity * 1.2] * 3, -1)
        layer = np.ones_like(f) * col
        return np.clip((f * (1 - a) + layer * a) * 255, 0, 255).astype(np.uint8)

    def _smoke_tex(self, h, w):
        s = np.zeros((h, w), np.float32)
        for sc in [4, 8, 16, 32, 64]:
            n = np.random.rand(max(h // sc + 1, 2), max(w // sc + 1, 2)).astype(np.float32)
            s += cv2.resize(n, (w, h), interpolation=cv2.INTER_CUBIC) * (sc / 64.)
        s = (s - s.min()) / (s.max() - s.min() + 1e-8)
        s = np.clip(s * 2., 0, 1)
        return cv2.GaussianBlur(np.power(s, 2.5), (41, 41), 0)

    def apply(self, image, cond):
        if cond == "fog":   return self.add_fog(image)
        if cond == "smoke": return self.add_smoke(image)
        return image.copy()  # clean

# ============================================================
# MAIN
# ============================================================
def main():
    degrader = DegradationPipeline(SEVERITY)

    for cond in CONDITIONS:
        os.makedirs(os.path.join(OUTPUT_ROOT, cond), exist_ok=True)

    data = json.load(open(UNIFIED_DATASET))
    samples = data["samples"]
    print(f"Images to process: {len(samples)}")
    print(f"Conditions: {CONDITIONS} | severity {SEVERITY}\n")

    done = 0
    missing = []
    already = 0

    for s in samples:
        fname = s["file_name"]  # COCO_train2014_XXXXXXXXXXXX.jpg
        src = os.path.join(COCO_SOURCE, fname)

        if not os.path.exists(src):
            missing.append(fname)
            continue

        img = cv2.imread(src)
        if img is None:
            missing.append(fname)
            continue

        for cond in CONDITIONS:
            out_path = os.path.join(OUTPUT_ROOT, cond, fname)
            if os.path.exists(out_path):
                already += 1
                continue
            deg = degrader.apply(img, cond)
            cv2.imwrite(out_path, deg)
        done += 1
        if done % 50 == 0:
            print(f"  {done}/{len(samples)} processed...")

    print("\n" + "=" * 50)
    print("DEGRADATION COMPLETE")
    print("=" * 50)
    print(f"  Images degraded : {done}")
    print(f"  Already existing (skipped) : {already}")
    print(f"  Not found : {len(missing)}")
    if missing:
        print("  Missing examples:")
        for m in missing[:5]:
            print(f"    {m}")
        print(f"  -> Check these images are in {COCO_SOURCE}")

    print(f"\nOutput: {OUTPUT_ROOT}/(clean|fog|smoke)/")
    print("\nNEXT STEP: Qwen3-VL calibration run on the 220 images")

    if missing:
        with open("./posture_missing_images.json", "w") as f:
            json.dump(missing, f, indent=2)
        print(f"\n  Missing list: posture_missing_images.json")

if __name__ == "__main__":
    main()

Images to process: 220
Conditions: ['clean', 'fog', 'smoke'] | severity 0.5

  50/220 processed...
  100/220 processed...
  150/220 processed...

DEGRADATION COMPLETE
  Images degraded : 179
  Already existing (skipped) : 0
  Not found : 41
  Missing examples:
    COCO_train2014_000000000002.jpg
    COCO_train2014_000000000003.jpg
    COCO_train2014_000000000004.jpg
    COCO_train2014_000000000005.jpg
    COCO_train2014_000000000006.jpg
  -> Check these images are in /Volumes/TheDay/thedayproject/Cours_Udem/cours_VL/HorusEye/Horus_dev/rq1_datasets/coco/images/train2014

Output: /Volumes/TheDay/thedayproject/Cours_Udem/cours_VL/HorusEye/Horus_dev/outputs/posture_degraded/(clean|fog|smoke)/

NEXT STEP: Qwen3-VL calibration run on the 220 images

  Missing list: posture_missing_images.json
